In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [2]:
num_files=15
file_prefix="D:\PHD\cgan\standalone\dataset/latent_representation_for_"
all_dfs = []

for i in range(1, num_files+1):
    df = pd.read_pickle(f"{file_prefix}{i}.pkl.zip", compression="zip")
    df = df[['x', 'y', 'z', 'time', 'latent_representation']]
    df['latent_representation'] = df['latent_representation'].apply(lambda x: x[0])
    all_dfs.append(df)

# Concatenate all dataframes
df_combined = pd.concat(all_dfs, ignore_index=True)
print(df_combined.head())
# Pivot table to get v as a matrix with shape (num_samples, 1200, 48)
df_pivot = df_combined.pivot_table(index=['x','y','z'], columns='time', values='latent_representation', aggfunc='first')


     x   y   z  time                              latent_representation
0 -117  79 -25     1  [0.50288206, -0.6009416, -0.16422167, 0.923491...
1 -113  79 -25     1  [-0.5462359, -0.47092205, -0.1574297, -0.10681...
2 -109  79 -25     1  [-0.3031874, -0.6288186, -0.5930966, -0.788726...
3 -105  79 -25     1  [0.36541444, 0.4556977, -0.7264514, -0.6433497...
4 -101  79 -25     1  [-0.9766785, 0.768204, -0.06921359, -0.8455191...


In [3]:
df_pivot.head()

time                                                         1   \
x    y   z                                                        
-117 -76 -25  [0.60152996, 0.98103255, -0.008870157, -0.2454...   
         -21  [0.096365936, 0.54300946, 0.8387079, 0.4684101...   
         -17  [0.6943935, 0.6303967, -0.8195155, 0.9442892, ...   
         -13  [0.56154877, 0.7658759, -0.31268513, 0.9745437...   
         -9   [0.9743955, -0.93269074, 0.027911449, 0.220510...   

time                                                         2   \
x    y   z                                                        
-117 -76 -25  [0.5743986, 0.7089401, 0.5678579, -0.93600154,...   
         -21  [-0.79866904, -0.12209187, 0.91154915, 0.76043...   
         -17  [0.9876632, -0.61406004, -0.3469003, 0.4697731...   
         -13  [-0.7751982, -0.325407, 0.4743654, -0.13019444...   
         -9   [-0.3710597, -0.19291241, 0.8153351, -0.286354...   

time                                                         3   \
x    y   z                                                        
-117 -76 -25  [-0.8775012, 0.77081597, 0.34914115, 0.9433937...   
         -21  [0.22371271, -0.7035653, -0.24191181, -0.04110...   
         -17  [-0.94120944, 0.9091235, 0.63393474, 0.7522126...   
         -13  [0.31826624, 0.65888816, 0.98721373, -0.370199...   
         -9   [-0.9283217, 0.25671372, 0.90663964, 0.91436, ...   

time                                                         4   \
x    y   z                                                        
-117 -76 -25  [-0.21666437, 0.276739, 0.67670155, -0.8597325...   
         -21  [-0.12166012, -0.90468264, -0.703625, -0.35087...   
         -17  [0.9533979, 0.374611, 0.08358036, 0.45856717, ...   
         -13  [-0.73936015, 0.13778922, -0.73885727, -0.4434...   
         -9   [-0.44492516, -0.2988802, -0.15240346, -0.3305...   

time                                                         5   \
x    y   z                                                        
-117 -76 -25  [0.36412022, -0.76997614, 0.266663, -0.4527933...   
         -21  [-0.6756259, 0.20125556, 0.41114074, -0.192610...   
         -17  [0.103173874, -0.9596432, -0.21567442, 0.48207...   
         -13  [0.95708394, 0.9988378, -0.84009784, -0.483040...   
         -9   [-0.5891063, -0.2228696, -0.18154225, -0.36551...   

time                                                         6   \
x    y   z                                                        
-117 -76 -25  [-0.3553231, -0.37004066, 0.40704593, -0.85804...   
         -21  [-0.2619261, -0.9286362, 0.87315434, 0.1708212...   
         -17  [0.8844934, -0.4611117, -0.35537854, -0.904041...   
         -13  [-0.43986663, -0.8859522, -0.60281193, -0.9730...   
         -9   [0.70254093, 0.91327983, 0.54900235, 0.8643656...   

time                                                         7   \
x    y   z                                                        
-117 -76 -25  [0.4651682, 0.32191125, -0.8944155, 0.7075796,...   
         -21  [0.12846749, -0.70874083, -0.09848327, 0.54393...   
         -17  [0.19190139, -0.2850105, 0.09330799, 0.2111620...   
         -13  [0.628395, 0.42571187, -0.22475213, -0.5347901...   
         -9   [-0.8378765, -0.8759353, 0.020267947, -0.99677...   

time                                                         8   \
x    y   z                                                        
-117 -76 -25  [-0.7426088, 0.7396835, -0.4338635, 0.6320076,...   
         -21  [-0.9192027, 0.39118025, 0.6679161, -0.7074012...   
         -17  [-0.55416745, 0.42543814, 0.32234558, -0.66634...   
         -13  [0.8604549, -0.80334365, -0.19985121, 0.601780...   
         -9   [-0.43921062, 0.9817179, 0.47819474, 0.8692322...   

time                                                         9   \
x    y   z                                                        
-117 -76 -25  [-0.16479653, 0.6672011, 0.44587672, -0.632495...   
         -21  [0.75533044, -0.95989746,

In [4]:
df_pivot.shape

(33600, 15)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [7]:
df_pivot_flat = df_pivot.reset_index()

In [8]:
df_pivot_flat.head()

time,x,y,z,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-117,-76,-25,"[0.60152996, 0.98103255, -0.008870157, -0.2454...","[0.5743986, 0.7089401, 0.5678579, -0.93600154,...","[-0.8775012, 0.77081597, 0.34914115, 0.9433937...","[-0.21666437, 0.276739, 0.67670155, -0.8597325...","[0.36412022, -0.76997614, 0.266663, -0.4527933...","[-0.3553231, -0.37004066, 0.40704593, -0.85804...","[0.4651682, 0.32191125, -0.8944155, 0.7075796,...","[-0.7426088, 0.7396835, -0.4338635, 0.6320076,...","[-0.16479653, 0.6672011, 0.44587672, -0.632495...","[-0.82881856, -0.4417005, -0.93003106, -0.8619...","[0.73687744, -0.4033748, -0.95862985, -0.47497...","[0.04219211, 0.5056695, -0.7162363, -0.4815756...","[-0.25694436, 0.4484407, -0.7467118, -0.507902...","[0.37512484, -0.87216157, -0.27912727, -0.6371...","[-0.8197715, -0.40555948, 0.9871446, -0.213071..."
1,-117,-76,-21,"[0.096365936, 0.54300946, 0.8387079, 0.4684101...","[-0.79866904, -0.12209187, 0.91154915, 0.76043...","[0.22371271, -0.7035653, -0.24191181, -0.04110...","[-0.12166012, -0.90468264, -0.703625, -0.35087...","[-0.6756259, 0.20125556, 0.41114074, -0.192610...","[-0.2619261, -0.9286362, 0.87315434, 0.1708212...","[0.12846749, -0.70874083, -0.09848327, 0.54393...","[-0.9192027, 0.39118025, 0.6679161, -0.7074012...","[0.75533044, -0.95989746, -0.90374166, -0.0630...","[0.23623419, 0.62348735, 0.38812354, 0.3865175...","[-0.7789519, 0.37796363, -0.87555295, -0.03135...","[0.7806267, 0.8975115, -0.5607745, -0.9473221,...","[0.95479465, 0.7803736, 0.1497075, -0.9292824,...","[-0.024801627, -0.3578735, -0.9100004, -0.1973...","[0.62667376, 0.6798731, 0.34012732, 0.5542102,..."
2,-117,-76,-17,"[0.6943935, 0.6303967, -0.8195155, 0.9442892, ...","[0.9876632, -0.61406004, -0.3469003, 0.4697731...","[-0.94120944, 0.9091235, 0.63393474, 0.7522126...","[0.9533979, 0.374611, 0.08358036, 0.45856717, ...","[0.103173874, -0.9596432, -0.21567442, 0.48207...","[0.8844934, -0.4611117, -0.35537854, -0.904041...","[0.19190139, -0.2850105, 0.09330799, 0.2111620...","[-0.55416745, 0.42543814, 0.32234558, -0.66634...","[0.6108569, 0.79253775, -0.007248677, 0.986103...","[-0.060510974, 0.8598963, -0.14712083, 0.79679...","[-0.16263914, 0.09215525, -0.3731518, 0.634308...","[-0.26758555, -0.94273233, 0.45466158, 0.84968...","[-0.60795283, -0.4063541, 0.5913332, -0.144027...","[-0.7996373, 0.7963824, 0.7067519, -0.98572326...","[0.8420596, 0.69856083, 0.15949044, 0.58392566..."
3,-117,-76,-13,"[0.56154877, 0.7658759, -0.31268513, 0.9745437...","[-0.7751982, -0.325407, 0.4743654, -0.13019444...","[0.31826624, 0.65888816, 0.98721373, -0.370199...","[-0.73936015, 0.13778922, -0.73885727, -0.4434...","[0.95708394, 0.9988378, -0.84009784, -0.483040...","[-0.43986663, -0.8859522, -0.60281193, -0.9730...","[0.628395, 0.42571187, -0.22475213, -0.5347901...","[0.8604549, -0.80334365, -0.19985121, 0.601780...","[0.62275165, 0.250471, -0.2056058, 0.48035592,...","[0.9358812, -0.9361252, -0.4325922, 0.56805587...","[0.5729953, 0.9068877, 0.5152233, 0.42906854, ...","[-0.7450801, 0.84187305, 0.71652573, -0.918943...","[-0.41168693, -0.66889644, 0.18859309, -0.2000...","[0.007128273, 0.85873616, 0.58662, -0.6822411,...","[0.17318805, -0.8831455, 0.7376071, 0.01548677..."
4,-117,-76,-9,"[0.9743955, -0.93269074, 0.027911449, 0.220510...","[-0.3710597, -0.19291241, 0.8153351, -0.286354...","[-0.9283217, 0.25671372, 0.90663964, 0.91436, ...","[-0.44492516, -0.2988802, -0.15240346, -0.3305...","[-0.5891063, -0.2228696, -0.18154225, -0.36551...","[0.70254093, 0.91327983, 0.54900235, 0.8643656...","[-0.8378765, -0.8759353, 0.020267947, -0.99677...","[-0.43921062, 0.9817179, 0.47819474, 0.8692322...","[0.975531, 0.67379576, 0.41823196, 0.63230884,...","[-0.6277772, 0.05303601, 0.62489265, -0.050239...","[0.97310853, -0.43587416, -0.16551034, -0.4941...","[0.66972435, 0.10993393, -0.88734424, 0.509057...","[0.17666648, 0.3667334, -0.90357393, -0.942890...","[-0.15619136, 0.62931603, -0.69244576, -0.7722...","[0.94355583, -0.54230475, 0.96

In [45]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, dataframe, sequence_length=5):
        self.dataframe = dataframe
        self.sequence_length = sequence_length
        self.num_samples = len(self.dataframe)
        self.max_time_frame = self.dataframe.columns[-1]

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        if idx >= self.num_samples:
            raise IndexError("Index out of range")

        # Retrieve spatial coordinates
        coordinates = self.dataframe.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)

        # Prepare the sequence of latent representations
        sequences = []
        num_time_window = self.max_time_frame // (self.sequence_length)

        for t in range(1, num_time_window+1):
            windows = []  
            for c in range(t, t + self.sequence_length):
                time_col = c
                if time_col > self.max_time_frame:
                    raise IndexError(f"Time column {time_col} not found in the dataframe")
            
                vector = self.dataframe[int(time_col)].iloc[idx]
                windows.append(vector)
                
            windows = np.stack(windows)
            sequences.append(windows)
        sequences = np.stack(sequences)
        # for t in range(self.sequence_length + 1):  # plus one for the target
        #     time_col = idx + t + 1
        #     if time_col > self.max_time_frame:
        #         raise IndexError(f"Time column {time_col} not found in the dataframe")
        #     if int(time_col) not in self.dataframe.columns:
        #         raise IndexError(f"Time column {time_col} not found in the dataframe")
        # 
        #     vector = self.dataframe[int(time_col)].iloc[idx]
        #     sequences.append(vector)

        # Convert the list of arrays into a 2D array

        # Split the sequences into source and target
        # src_seq = sequences[:-1]  # All but the last
        # tgt_seq = sequences[-1]  # Only the last

        # Combine the spatial coordinates with the source sequence
        # src_seq = np.concatenate([np.tile(coordinates, (self.sequence_length, 1)), src_seq], axis=1)

        # return torch.tensor(src_seq, dtype=torch.float), torch.tensor(tgt_seq, dtype=torch.float)
        return coordinates, torch.tensor(sequences, dtype=torch.float)


In [46]:
dataset = SpatioTemporalDataset(df_pivot_flat, 4)

In [47]:
a, b = dataset.__getitem__(0)

In [48]:
b.shape

torch.Size([3, 5, 48])

In [56]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, drop_last=True)

In [57]:
i=0
for coord, seq in dataloader:
    print(seq.shape)
    for seq1 in seq:
        for s in seq1:
            src_seq = s[:-1]
            tgt_seq = s[-1]
            print(src_seq.shape)
            print(tgt_seq.shape)
    if i==1:
        break
    i+=1

torch.Size([32, 3, 5, 48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size([4, 48])
torch.Size([48])
torch.Size(

In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math

In [60]:

class TransformerModel(nn.Module):
    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):
        super(TransformerModel, self).__init__()
        from torch.nn import Transformer
        self.model_type = 'Transformer'
        self.pos_encoder = PositionalEncoding(ninp, dropout)
        self.encoder = nn.Embedding(ntoken, ninp)
        self.transformer = Transformer(d_model=ninp, nhead=nhead, 
                                       num_encoder_layers=nlayers,
                                       num_decoder_layers=nlayers,
                                       dim_feedforward=nhid, dropout=dropout)
        self.ninp = ninp
        self.decoder = nn.Linear(ninp, ntoken)

    def forward(self, src, tgt, src_mask, tgt_mask, memory_mask, 
                src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask):
        src = self.encoder(src) * math.sqrt(self.ninp)
        src = self.pos_encoder(src)
        tgt = self.encoder(tgt) * math.sqrt(self.ninp)
        tgt = self.pos_encoder(tgt)
        output = self.transformer(src, tgt, src_mask, tgt_mask, memory_mask,
                                  src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask)
        output = self.decoder(output)
        return output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)


In [ ]:
def train(model, dataloader, optimizer, criterion, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for source, target in dataloader:
            optimizer.zero_grad()
            output = model(source, target, src_mask, tgt_mask, memory_mask, 
                           src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask)
            loss = criterion(output.view(-1, ntokens), target.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch}, Loss: {total_loss / len(dataloader)}')




In [ ]:
# Example usage
# model = TransformerModel(ntokens, ninp, nhead, nhid, nlayers, dropout)
# optimizer = optim.SGD(model.parameters(), lr=0.01)
# criterion = nn.CrossEntropyLoss()
# train(model, dataloader, optimizer, criterion, epochs)